## Initializing polymer systems using a DPD potential
This notebook walks through the PhantomWalk functions for packing linear polymers in a box. The polymers are first placed in a cubic box using a random walk. Then a short HOOMD simulation is run with the soft force potential of Dissipative Particle Dynamics. The simulation ends when the pair energy from the DPD potential reaches a stable state.

In [ ]:
import sys
import os
sys.path.append('../lib/')
import create_system_dpd
from create_system_dpd import create_polymer_system_dpd
import matplotlib
import numpy as np  
import gsd, gsd.hoomd 
import hoomd 
import time
import freud
import matplotlib_inline
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.style.use("ggplot")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
import warnings
warnings.filterwarnings("ignore")

In [ ]:
num_pol=100
num_mon=100
N = num_pol*num_mon
density=1.1
A=50000
r_cut=1.05
min_pair_dist=0.9
last_dpd_frame, closest, s, e_cut = create_polymer_system_dpd(
    num_pol=num_pol,
    num_mon=num_mon,
    density=density,
    k=50000,
    r_cut=r_cut,
    A=A,
    gamma=1200,
    sim_seed=1234,
    np_seed=1234,
    min_pair_dist=min_pair_dist,
    energy_scaling = 4,
)
print(f"Finished in time = {s:.2f}s")

## Visualize the Energy

In [ ]:
log = np.genfromtxt("log.txt", names=True)
pe = log["mdcomputeThermodynamicQuantitiespotential_energy"]
pairs = log["mdpairDPDenergy"]
bonds = log["mdbondHarmonicenergy"]
print("Total steps",len(pe)*100+100)
print("DPD Energy Cutoff= ",e_cut)

slice_idx = 0
short_idx= None
x_values = range(slice_idx, len(pe[:short_idx]))
plt.plot(x_values,pe[slice_idx:short_idx]/N, label="potential energy")
plt.plot(x_values,pairs[slice_idx:short_idx]/N, label="DPD pair energy")
plt.hlines(e_cut,xmin=slice_idx,xmax=len(pairs[:short_idx]),color="blue",linestyle="--",label="Calculated Pair Energy Cutoff")
plt.title("Energy vs Time")
plt.xlabel("Frame (time/log_freq)")
plt.ylabel("Energy")

plt.legend()
#plt.savefig("10-10mers-energy-cut")

In [ ]:
plt.legend()
plt.plot(bonds, label="bond energy")
plt.title("Bond Energy vs Time")
plt.xlabel("Frame (time/log_freq)")
plt.ylabel("Energy")
plt.legend()

In [ ]:
rdf_data = np.genfromtxt("rdf.csv", delimiter=",")
plt.plot(rdf_data[:, 0], rdf_data[:, 1])
plt.title("Radial Distribution Function")
plt.xlabel("$r$")
plt.ylabel("$g(r)$")
plt.show()